In [ ]:
# ============================================================
# CELL 2: Extract image ZIP
# ============================================================

import os
import zipfile

ZIP_PATH = "/content/JSRT_Balanced_54_54.zip"  # change this

EXTRACT_DIR = "/content/jsrt_images"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

Extracted to: /content/jsrt_images


In [ ]:
import os
import zipfile
import random
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (300, 300)
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-5

CLASS0_NAME = "Benign"
CLASS1_NAME = "Malignant"

OUTPUT_ROOT = "/content/JSRT_BT_LL_Results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: []


In [ ]:
def find_dataset_root(base_dir):
    for root, dirs, files in os.walk(base_dir):
        if CLASS0_NAME in dirs and CLASS1_NAME in dirs:
            return root
    return None

DATASET_ROOT = find_dataset_root(EXTRACT_DIR)

print("Dataset root:", DATASET_ROOT)

BENIGN_DIR = os.path.join(DATASET_ROOT, CLASS0_NAME)
MALIGNANT_DIR = os.path.join(DATASET_ROOT, CLASS1_NAME)

print("Benign count:", len(os.listdir(BENIGN_DIR)))
print("Malignant count:", len(os.listdir(MALIGNANT_DIR)))

Dataset root: /content/jsrt_images
Benign count: 54
Malignant count: 54


In [ ]:
image_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

data = []

for fname in os.listdir(BENIGN_DIR):
    if fname.lower().endswith(image_extensions):
        data.append({
            "filepath": os.path.join(BENIGN_DIR, fname),
            "label": 0,
            "class_name": "Benign"
        })

for fname in os.listdir(MALIGNANT_DIR):
    if fname.lower().endswith(image_extensions):
        data.append({
            "filepath": os.path.join(MALIGNANT_DIR, fname),
            "label": 1,
            "class_name": "Malignant"
        })

df = pd.DataFrame(data)

print(df.head())
print(df["class_name"].value_counts())
print("Total images:", len(df))

                                   filepath  label class_name
0  /content/jsrt_images/Benign/JPCLN002.jpg      0     Benign
1  /content/jsrt_images/Benign/JPCLN101.jpg      0     Benign
2  /content/jsrt_images/Benign/JPCLN122.jpg      0     Benign
3  /content/jsrt_images/Benign/JPCLN141.jpg      0     Benign
4  /content/jsrt_images/Benign/JPCLN137.jpg      0     Benign
class_name
Benign       54
Malignant    54
Name: count, dtype: int64
Total images: 108


In [ ]:
def make_dataset_from_df(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(
            img,
            channels=3,
            expand_animations=False
        )
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)   # keep 0–255
        return img, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
# ============================================================
# Lag–Lead preprocessing layer
# Run this BEFORE build_jsrt_model()
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers

class LagLeadPreprocessing(layers.Layer):
    def __init__(self, T1=0.2, T2=0.1, **kwargs):
        super().__init__(**kwargs)
        self.T1 = T1
        self.T2 = T2

        gaussian = tf.constant(
            [[1., 2., 1.],
             [2., 4., 2.],
             [1., 2., 1.]], dtype=tf.float32
        ) / 16.0

        laplacian = tf.constant(
            [[0., -1., 0.],
             [-1., 4., -1.],
             [0., -1., 0.]], dtype=tf.float32
        )

        self.gaussian = tf.reshape(gaussian, [3, 3, 1, 1])
        self.laplacian = tf.reshape(laplian if False else laplacian, [3, 3, 1, 1])

    def call(self, x):
        gaussian_rgb = tf.tile(self.gaussian, [1, 1, 3, 1])
        laplacian_rgb = tf.tile(self.laplacian, [1, 1, 3, 1])

        smooth = tf.nn.depthwise_conv2d(
            x,
            gaussian_rgb,
            strides=[1, 1, 1, 1],
            padding="SAME"
        )

        lag = (1.0 - self.T1) * x + self.T1 * smooth

        edge = tf.nn.depthwise_conv2d(
            lag,
            laplacian_rgb,
            strides=[1, 1, 1, 1],
            padding="SAME"
        )

        lead = lag + self.T2 * edge

        return tf.clip_by_value(lead, 0.0, 1.0)

In [ ]:
# ============================================================
# CELL 8: Build JSRT Lag-Lead EfficientNetB3 model
# ============================================================

def build_jsrt_model(
    input_shape=(300, 300, 3),
    T1=0.2,
    T2=0.1,
    drop_rate=0.5
):
    inputs = layers.Input(shape=input_shape)

    # Data augmentation for small dataset
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.10)(x)

    # Normalize 0–255 to 0–1
    x = layers.Rescaling(1.0 / 255.0)(x)

    # Proposed Lag–Lead preprocessing
    x = LagLeadPreprocessing(T1=T1, T2=T2, name="lag_lead_preprocessing")(x)

    # Convert back to 0–255 before EfficientNetB3
    x = layers.Lambda(lambda t: t * 255.0, name="back_to_0_255")(x)

    # EfficientNetB3 backbone
    base_model = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    # Freeze backbone first
    base_model.trainable = False

    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dropout(drop_rate)(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(drop_rate)(x)

    outputs = layers.Dense(1, activation="sigmoid", name="classification_output")(x)

    model = Model(inputs, outputs, name="JSRT_LagLead_EfficientNetB3")

    return model

In [ ]:
tf.keras.backend.clear_session()

model = build_jsrt_model(
    input_shape=(300, 300, 3),
    T1=0.2,
    T2=0.1,
    drop_rate=0.5
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "JSRT_LagLead_EfficientNetB3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 300, 300, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 300, 300, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lag_lead_preprocessing          │ (None, 300, 300, 3)    │             0 │
│ (LagLeadPreprocessing)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ back_to_0_255 (Lambda)          │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       196,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_output (Dense)   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,986,544 (41.91 MB)

 Trainable params: 199,937 (781.00 KB)

 Non-trainable params: 10,786,607 (41.15 MB)

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:")
print(train_df["class_name"].value_counts())

print("\nValidation:")
print(val_df["class_name"].value_counts())

print("\nTest:")
print(test_df["class_name"].value_counts())

Train:
class_name
Malignant    38
Benign       37
Name: count, dtype: int64

Validation:
class_name
Benign       8
Malignant    8
Name: count, dtype: int64

Test:
class_name
Benign       9
Malignant    8
Name: count, dtype: int64


In [ ]:
train_ds = make_dataset_from_df(train_df, shuffle=True)
val_ds = make_dataset_from_df(val_df, shuffle=False)
test_ds = make_dataset_from_df(test_df, shuffle=False)

for images, labels in train_ds.take(1):
    print("Image shape:", images.shape)
    print("Labels:", labels.numpy())
    print("Min pixel:", tf.reduce_min(images).numpy())
    print("Max pixel:", tf.reduce_max(images).numpy())

Image shape: (4, 300, 300, 3)
Labels: [0. 0. 1. 0.]
Min pixel: 0.0
Max pixel: 255.0


In [ ]:
# ============================================================
# 5-FOLD CROSS-VALIDATION
# ============================================================

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

X = df["filepath"].values
y = df["label"].values

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    train_fold_df = df.iloc[train_idx].reset_index(drop=True)
    val_fold_df = df.iloc[val_idx].reset_index(drop=True)

    print("Train distribution:")
    print(train_fold_df["class_name"].value_counts())

    print("Validation distribution:")
    print(val_fold_df["class_name"].value_counts())

    train_fold_ds = make_dataset_from_df(train_fold_df, shuffle=True)
    val_fold_ds = make_dataset_from_df(val_fold_df, shuffle=False)

    tf.keras.backend.clear_session()

    fold_model = build_jsrt_model(
        input_shape=(300, 300, 3),
        T1=0.2,
        T2=0.1,
        drop_rate=0.5
    )

    fold_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    fold_output_dir = os.path.join(OUTPUT_ROOT, f"fold_{fold}")
    os.makedirs(fold_output_dir, exist_ok=True)

    fold_callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(fold_output_dir, "best_model.keras"),
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            os.path.join(fold_output_dir, "training_log.csv")
        )
    ]

    start_time = time.time()

    history_fold = fold_model.fit(
        train_fold_ds,
        validation_data=val_fold_ds,
        epochs=EPOCHS,
        callbacks=fold_callbacks,
        verbose=1
    )

    training_time = time.time() - start_time

    y_true = []
    y_prob = []

    for images, labels in val_fold_ds:
        probs = fold_model.predict(images, verbose=0)
        y_prob.extend(probs.ravel())
        y_true.extend(labels.numpy().ravel())

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)

    print(f"\nFold {fold} Metrics")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1-score :", f1)
    print("AUC      :", auc)
    print("Training time:", training_time, "seconds")
    print("Confusion Matrix:")
    print(cm)

    fold_results.append({
        "Fold": fold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1_score": f1,
        "AUC": auc,
        "Training_time_sec": training_time,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })


FOLD 1/5
Train distribution:
class_name
Benign       43
Malignant    43
Name: count, dtype: int64
Validation distribution:
class_name
Benign       11
Malignant    11
Name: count, dtype: int64
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 993ms/step - accuracy: 0.5745 - auc: 0.5948 - loss: 1.2936 - precision: 0.6469 - recall: 0.4229
Epoch 1: val_auc improved from None to 0.57025, saving model to /content/JSRT_BT_LL_Results/fold_1/best_model.keras

Epoch 1: finished saving model to /content/JSRT_BT_LL_Results/fold_1/best_model.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - accuracy: 0.5233 - auc: 0.5357 - loss: 1.2643 - precision: 0.5333 - recall: 0.3721 - val_accuracy: 0.5000 - val_auc: 0.5702 - val_loss: 0.7419 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-05
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 968ms/step - accuracy: 0.4916 - auc: 0.4032 - loss: 1.6350 - precision: 0.5209 - recall: 0.2631
Epoch 2: val_auc improved from 0.57025 to 0.57438, saving model t


Fold 3 Metrics
Accuracy : 0.7272727272727273
Precision: 0.7272727272727273
Recall   : 0.7272727272727273
F1-score : 0.7272727272727273
AUC      : 0.7851239669421488
Training time: 331.5552246570587 seconds
Confusion Matrix:
[[8 3]
 [3 8]]

FOLD 4/5
Train distribution:
class_name
Malignant    44
Benign       43
Name: count, dtype: int64
Validation distribution:
class_name
Benign       11
Malignant    10
Name: count, dtype: int64
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 947ms/step - accuracy: 0.4301 - auc: 0.4362 - loss: 1.2026 - precision: 0.3741 - recall: 0.3433
Epoch 1: val_auc improved from None to 0.22727, saving model to /content/JSRT_BT_LL_Results/fold_4/best_model.keras

Epoch 1: finished saving model to /content/JSRT_BT_LL_Results/fold_4/best_model.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - accuracy: 0.3908 - auc: 0.4125 - loss: 1.2481 - precision: 0.3953 - recall: 0.3864 - val_accuracy: 0.3333 - val_auc: 0.2273 - val_loss: 0.7841 - val_precision: 0.0000e+00 - val_recall: 0


Fold 4 Metrics
Accuracy : 0.5714285714285714
Precision: 0.5384615384615384
Recall   : 0.7
F1-score : 0.6086956521739131
AUC      : 0.5
Training time: 620.3284647464752 seconds
Confusion Matrix:
[[5 6]
 [3 7]]

FOLD 5/5
Train distribution:
class_name
Benign       44
Malignant    43
Name: count, dtype: int64
Validation distribution:
class_name
Malignant    11
Benign       10
Name: count, dtype: int64
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 979ms/step - accuracy: 0.4497 - auc: 0.3653 - loss: 1.6260 - precision: 0.1975 - recall: 0.0558
Epoch 1: val_auc improved from None to 0.26818, saving model to /content/JSRT_BT_LL_Results/fold_5/best_model.keras

Epoch 1: finished saving model to /content/JSRT_BT_LL_Results/fold_5/best_model.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - accuracy: 0.4713 - auc: 0.4297 - loss: 1.5979 - precision: 0.3636 - recall: 0.0930 - val_accuracy: 0.4762 - val_auc: 0.2682 - val_loss: 0.9020 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0

In [ ]:
# ============================================================
# 5-FOLD MEAN ± SD METRICS TABLE
# Run this AFTER the 5-fold loop finishes
# ============================================================

import pandas as pd
import numpy as np
import os

# Convert fold results to DataFrame
cv_results_df = pd.DataFrame(fold_results)

print("Per-fold results:")
print(cv_results_df)

# Metrics to summarize
metric_cols = ["Accuracy", "Precision", "Recall", "F1_score", "AUC"]

summary_rows = []

for metric in metric_cols:
    mean_value = cv_results_df[metric].mean()
    std_value = cv_results_df[metric].std()

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "SD": std_value,
        "Mean ± SD": f"{mean_value:.4f} ± {std_value:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)

print("\n5-Fold Cross-Validation Mean ± SD:")
print(summary_df)

Per-fold results:
   Fold  Accuracy  Precision    Recall  F1_score       AUC  Training_time_sec  \
0     1  0.454545   0.000000  0.000000  0.000000  0.578512         335.633053   
1     2  0.454545   0.000000  0.000000  0.000000  0.570248         227.144021   
2     3  0.727273   0.727273  0.727273  0.727273  0.785124         331.555225   
3     4  0.571429   0.538462  0.700000  0.608696  0.500000         620.328465   
4     5  0.476190   0.000000  0.000000  0.000000  0.590909         665.733122   

   TN  FP  FN  TP  
0  10   1  11   0  
1  10   1  11   0  
2   8   3   3   8  
3   5   6   3   7  
4  10   0  11   0  

5-Fold Cross-Validation Mean ± SD:
      Metric      Mean        SD        Mean ± SD
0   Accuracy  0.536797  0.116923  0.5368 ± 0.1169
1  Precision  0.253147  0.353005  0.2531 ± 0.3530
2     Recall  0.285455  0.390994  0.2855 ± 0.3910
3   F1_score  0.267194  0.368264  0.2672 ± 0.3683
4        AUC  0.604959  0.106744  0.6050 ± 0.1067


In [ ]:
print("Probability min:", y_prob.min())
print("Probability max:", y_prob.max())
print("Probability mean:", y_prob.mean())
print("First 20 probabilities:", y_prob[:20])

Probability min: 0.037442047
Probability max: 0.39611155
Probability mean: 0.17157267
First 20 probabilities: [0.22739328 0.1384779  0.3788581  0.13783038 0.1409999  0.1197409
 0.11829974 0.04826892 0.25138918 0.07293323 0.20502691 0.1644612
 0.3396552  0.03744205 0.12608302 0.05174392 0.16494685 0.39611155
 0.16181375 0.18075253]


In [ ]:
# ============================================================
# Threshold tuning based on F1-score
# ============================================================

best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.05, 0.51, 0.01):
    y_pred_temp = (y_prob >= threshold).astype(int)
    f1_temp = f1_score(y_true, y_pred_temp, zero_division=0)

    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

print("Probability min:", y_prob.min())
print("Probability max:", y_prob.max())
print("Probability mean:", y_prob.mean())
print("Best threshold:", best_threshold)
print("Best F1 at threshold:", best_f1)

y_pred = (y_prob >= best_threshold).astype(int)

Probability min: 0.037442047
Probability max: 0.39611155
Probability mean: 0.17157267
Best threshold: 0.14
Best F1 at threshold: 0.6956521739130435


In [ ]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    auc = roc_auc_score(y_true, y_prob)
except:
    auc = np.nan

cm = confusion_matrix(y_true, y_pred)

In [ ]:
fold_results.append({
    "Fold": fold,
    "Best_threshold": best_threshold,
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1_score": f1,
    "AUC": auc,
    "Training_time_sec": training_time,
    "TN": cm[0, 0],
    "FP": cm[0, 1],
    "FN": cm[1, 0],
    "TP": cm[1, 1]
})

In [ ]:
# ============================================================
# RUN ONE FOLD AT A TIME
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np
import pandas as pd
import time
import os
import gc
import tensorflow as tf

N_SPLITS = 5
FOLD_TO_RUN = 5  # CHANGE: 1, 2, 3, 4, or 5

EPOCHS = 15
BATCH_SIZE = 2
LEARNING_RATE = 1e-5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

X = df["filepath"].values
y = df["label"].values

splits = list(skf.split(X, y))

train_idx, val_idx = splits[FOLD_TO_RUN - 1]

print("=" * 70)
print(f"RUNNING FOLD {FOLD_TO_RUN}/{N_SPLITS}")
print("=" * 70)

train_fold_df = df.iloc[train_idx].reset_index(drop=True)
val_fold_df = df.iloc[val_idx].reset_index(drop=True)

print("Train distribution:")
print(train_fold_df["class_name"].value_counts())

print("\nValidation distribution:")
print(val_fold_df["class_name"].value_counts())

train_fold_ds = make_dataset_from_df(train_fold_df, shuffle=True)
val_fold_ds = make_dataset_from_df(val_fold_df, shuffle=False)

tf.keras.backend.clear_session()

fold_model = build_jsrt_model(
    input_shape=(300, 300, 3),
    T1=0.2,
    T2=0.1,
    drop_rate=0.5
)

fold_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

fold_output_dir = os.path.join(OUTPUT_ROOT, f"fold_{FOLD_TO_RUN}")
os.makedirs(fold_output_dir, exist_ok=True)

fold_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(fold_output_dir, "best_model.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        os.path.join(fold_output_dir, "training_log.csv")
    )
]

start_time = time.time()

history_fold = fold_model.fit(
    train_fold_ds,
    validation_data=val_fold_ds,
    epochs=EPOCHS,
    callbacks=fold_callbacks,
    verbose=1
)

training_time = time.time() - start_time

# ============================================================
# Prediction
# ============================================================

y_true = []
y_prob = []

for images, labels in val_fold_ds:
    probs = fold_model.predict(images, verbose=0)
    y_prob.extend(probs.ravel())
    y_true.extend(labels.numpy().ravel())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)

print("Probability min:", y_prob.min())
print("Probability max:", y_prob.max())
print("Probability mean:", y_prob.mean())

# ============================================================
# Threshold tuning
# ============================================================

best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.05, 0.51, 0.01):
    y_pred_temp = (y_prob >= threshold).astype(int)
    f1_temp = f1_score(y_true, y_pred_temp, zero_division=0)

    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best F1 at threshold:", best_f1)

y_pred = (y_prob >= best_threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    auc = roc_auc_score(y_true, y_prob)
except:
    auc = np.nan

cm = confusion_matrix(y_true, y_pred)

print("\nFold Metrics")
print("Fold:", FOLD_TO_RUN)
print("Best threshold:", best_threshold)
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)
print("AUC      :", auc)
print("Training time:", training_time, "seconds")
print("Confusion Matrix:")
print(cm)

fold_result = pd.DataFrame([{
    "Fold": FOLD_TO_RUN,
    "Best_threshold": best_threshold,
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1_score": f1,
    "AUC": auc,
    "Training_time_sec": training_time,
    "TN": cm[0, 0],
    "FP": cm[0, 1],
    "FN": cm[1, 0],
    "TP": cm[1, 1]
}])

fold_result_path = os.path.join(fold_output_dir, f"fold_{FOLD_TO_RUN}_result.csv")
fold_result.to_csv(fold_result_path, index=False)

print("Saved fold result:", fold_result_path)

# Clear memory
del fold_model
del train_fold_ds
del val_fold_ds
gc.collect()
tf.keras.backend.clear_session()

RUNNING FOLD 5/5
Train distribution:
class_name
Benign       44
Malignant    43
Name: count, dtype: int64

Validation distribution:
class_name
Malignant    11
Benign       10
Name: count, dtype: int64
Epoch 1/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 624ms/step - accuracy: 0.4282 - auc: 0.4189 - loss: 1.2259 - precision: 0.4092 - recall: 0.4546
Epoch 1: val_auc improved from None to 0.71818, saving model to /content/JSRT_BT_LL_Results/fold_5/best_model.keras

Epoch 1: finished saving model to /content/JSRT_BT_LL_Results/fold_5/best_model.keras
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 907ms/step - accuracy: 0.4943 - auc: 0.4725 - loss: 1.1018 - precision: 0.4894 - recall: 0.5349 - val_accuracy: 0.5238 - val_auc: 0.7182 - val_loss: 0.6983 - val_precision: 0.5238 - val_recall: 1.0000 - learning_rate: 1.0000e-05
Epoch 2/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 509ms/step - accuracy: 0.4442 - auc: 0.4461 - loss: 1.3104 - precision: 0.4470 - recall: 0.4925
Epoch 2: val_auc did not improve from 0.71818
44/44 ━━━━━━━━━━

In [ ]:
import pandas as pd
import os
from glob import glob

result_files = sorted(glob(os.path.join(OUTPUT_ROOT, "fold_*", "fold_*_result.csv")))

print("Found result files:")
print(result_files)

all_results = []

for f in result_files:
    all_results.append(pd.read_csv(f))

cv_results_df = pd.concat(all_results, ignore_index=True)

print("All fold results:")
print(cv_results_df)

metric_cols = ["Accuracy", "Precision", "Recall", "F1_score", "AUC"]

summary_rows = []

for metric in metric_cols:
    mean_value = cv_results_df[metric].mean()
    std_value = cv_results_df[metric].std()

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "SD": std_value,
        "Mean ± SD": f"{mean_value:.4f} ± {std_value:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)

print("\n5-Fold Mean ± SD:")
print(summary_df)

summary_path = os.path.join(OUTPUT_ROOT, "jsrt_5fold_mean_sd_metrics.csv")
cv_results_path = os.path.join(OUTPUT_ROOT, "jsrt_5fold_results.csv")

summary_df.to_csv(summary_path, index=False)
cv_results_df.to_csv(cv_results_path, index=False)

print("Saved:", summary_path)
print("Saved:", cv_results_path)

Found result files:
['/content/JSRT_BT_LL_Results/fold_1/fold_1_result.csv', '/content/JSRT_BT_LL_Results/fold_2/fold_2_result.csv', '/content/JSRT_BT_LL_Results/fold_3/fold_3_result.csv', '/content/JSRT_BT_LL_Results/fold_4/fold_4_result.csv', '/content/JSRT_BT_LL_Results/fold_5/fold_5_result.csv']
All fold results:
   Fold  Best_threshold  Accuracy  Precision    Recall  F1_score       AUC  \
0     1            0.17  0.590909   0.555556  0.909091  0.689655  0.636364   
1     2            0.44  0.590909   0.550000  1.000000  0.709677  0.735537   
2     3            0.31  0.545455   0.523810  1.000000  0.687500  0.380165   
3     4            0.13  0.523810   0.500000  1.000000  0.666667  0.545455   
4     5            0.05  0.523810   0.523810  1.000000  0.687500  0.709091   

   Training_time_sec  TN  FP  FN  TP  
0         475.814908   3   8   1  10  
1         259.804276   2   9   0  11  
2         283.625075   1  10   0  11  
3         482.643773   1  10   0  10  
4         197.739

In [ ]:
# ============================================================
# CELL 2: Extract image ZIP
# ============================================================

import os
import zipfile

ZIP_PATH = "/JSRT_Benign_Malignant.zip"  # change this

EXTRACT_DIR = "/content/JSRT_images"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

Extracted to: /content/JSRT_images


In [ ]:
def find_dataset_root(base_dir):
    for root, dirs, files in os.walk(base_dir):
        if CLASS0_NAME in dirs and CLASS1_NAME in dirs:
            return root
    return None

DATASET_ROOT = find_dataset_root(EXTRACT_DIR)

print("Dataset root:", DATASET_ROOT)

BENIGN_DIR = os.path.join(DATASET_ROOT, CLASS0_NAME)
MALIGNANT_DIR = os.path.join(DATASET_ROOT, CLASS1_NAME)

print("Benign count:", len(os.listdir(BENIGN_DIR)))
print("Malignant count:", len(os.listdir(MALIGNANT_DIR)))

Dataset root: /content/JSRT_images
Benign count: 54
Malignant count: 100


In [ ]:
image_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

data = []

for fname in os.listdir(BENIGN_DIR):
    if fname.lower().endswith(image_extensions):
        data.append({
            "filepath": os.path.join(BENIGN_DIR, fname),
            "label": 0,
            "class_name": "Benign"
        })

for fname in os.listdir(MALIGNANT_DIR):
    if fname.lower().endswith(image_extensions):
        data.append({
            "filepath": os.path.join(MALIGNANT_DIR, fname),
            "label": 1,
            "class_name": "Malignant"
        })

df = pd.DataFrame(data)

print(df.head())
print(df["class_name"].value_counts())
print("Total images:", len(df))

                                   filepath  label class_name
0  /content/JSRT_images/Benign/JPCLN002.jpg      0     Benign
1  /content/JSRT_images/Benign/JPCLN101.jpg      0     Benign
2  /content/JSRT_images/Benign/JPCLN122.jpg      0     Benign
3  /content/JSRT_images/Benign/JPCLN141.jpg      0     Benign
4  /content/JSRT_images/Benign/JPCLN137.jpg      0     Benign
class_name
Malignant    100
Benign        54
Name: count, dtype: int64
Total images: 154


In [ ]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

SEED = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation distribution:")
print(val_df["class_name"].value_counts())

print("\nTest distribution:")
print(test_df["class_name"].value_counts())

Train distribution:
class_name
Malignant    69
Benign       38
Name: count, dtype: int64

Validation distribution:
class_name
Malignant    15
Benign        8
Name: count, dtype: int64

Test distribution:
class_name
Malignant    16
Benign        8
Name: count, dtype: int64


In [ ]:
# ============================================================
# CLASS WEIGHTS FROM TRAINING SET ONLY
# ============================================================

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"].values)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)

class_weight_dict = dict(zip(classes, class_weights_array))

print("Class weights:")
print(class_weight_dict)

Class weights:
{np.int64(0): np.float64(1.4078947368421053), np.int64(1): np.float64(0.7753623188405797)}


In [ ]:
# ============================================================
# DATASET FUNCTION
# ============================================================

IMG_SIZE = (300, 300)
BATCH_SIZE = 2

def make_dataset_from_df(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(
            img,
            channels=3,
            expand_animations=False
        )
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)  # keep 0-255
        return img, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = make_dataset_from_df(train_df, shuffle=True)
val_ds = make_dataset_from_df(val_df, shuffle=False)
test_ds = make_dataset_from_df(test_df, shuffle=False)

In [ ]:
# ============================================================
# TRAIN WITH CLASS WEIGHTS
# ============================================================

import os
import time
import tensorflow as tf

OUTPUT_ROOT = "/content/JSRT_ClassWeight_Results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

EPOCHS = 30
LEARNING_RATE = 1e-5

tf.keras.backend.clear_session()

model = build_jsrt_model(
    input_shape=(300, 300, 3),
    T1=0.2,
    T2=0.1,
    drop_rate=0.5
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_ROOT, "best_model.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        os.path.join(OUTPUT_ROOT, "training_log.csv")
    )
]

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

training_time = time.time() - start_time

print("Training time:", training_time, "seconds")

Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - accuracy: 0.6178 - auc: 0.6524 - loss: 0.9169 - precision: 0.7808 - recall: 0.5757
Epoch 1: val_auc improved from None to 0.33333, saving model to /content/JSRT_ClassWeight_Results/best_model.keras

Epoch 1: finished saving model to /content/JSRT_ClassWeight_Results/best_model.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 62s 744ms/step - accuracy: 0.5140 - auc: 0.5229 - loss: 1.1506 - precision: 0.6545 - recall: 0.5217 - val_accuracy: 0.3913 - val_auc: 0.3333 - val_loss: 0.7507 - val_precision: 0.5333 - val_recall: 0.5333 - learning_rate: 1.0000e-05
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.5189 - auc: 0.4477 - loss: 1.0975 - precision: 0.6667 - recall: 0.6006
Epoch 2: val_auc improved from 0.33333 to 0.35000, saving model to /content/JSRT_ClassWeight_Results/best_model.keras

Epoch 2: finished saving model to /content/JSRT_ClassWeight_Results/best_model.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 77s 648ms/step - accuracy: 0.4673 

In [ ]:
# ============================================================
# TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_true = []
y_prob = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_prob.extend(probs.ravel())
    y_true.extend(labels.numpy().ravel())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)

print("Probability min:", y_prob.min())
print("Probability max:", y_prob.max())
print("Probability mean:", y_prob.mean())

Probability min: 0.06305374
Probability max: 0.85256445
Probability mean: 0.3817548


In [ ]:
# ============================================================
# FIND BEST THRESHOLD USING VALIDATION SET
# ============================================================

val_true = []
val_prob = []

for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    val_prob.extend(probs.ravel())
    val_true.extend(labels.numpy().ravel())

val_true = np.array(val_true).astype(int)
val_prob = np.array(val_prob)

best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.05, 0.51, 0.01):
    val_pred_temp = (val_prob >= threshold).astype(int)
    f1_temp = f1_score(val_true, val_pred_temp, zero_division=0)

    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

print("Best validation threshold:", best_threshold)
print("Best validation F1:", best_f1)

Best validation threshold: 0.05
Best validation F1: 0.7777777777777778


In [ ]:
# ============================================================
# APPLY VALIDATION THRESHOLD TO TEST SET
# ============================================================

y_pred = (y_prob >= best_threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    auc = roc_auc_score(y_true, y_prob)
except:
    auc = np.nan

cm = confusion_matrix(y_true, y_pred)

print("Test Metrics")
print("Best threshold:", best_threshold)
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)
print("AUC      :", auc)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(cm)

Test Metrics
Best threshold: 0.05
Accuracy : 0.6666666666666666
Precision: 0.6666666666666666
Recall   : 1.0
F1-score : 0.8
AUC      : 0.2578125

Classification Report:
              precision    recall  f1-score   support

      Benign       0.00      0.00      0.00         8
   Malignant       0.67      1.00      0.80        16

    accuracy                           0.67        24
   macro avg       0.33      0.50      0.40        24
weighted avg       0.44      0.67      0.53        24


Confusion Matrix:
[[ 0  8]
 [ 0 16]]
